# Bisaya TDNN-HMM Training (Kaldi)

Implements Sections 5-9 of `Bisaya_Filipino_ASR_Reproduction_Guide.md`: an HMM-GMM
baseline followed by a TDNN-HMM hybrid, trained from scratch with
[Kaldi](https://github.com/kaldi-asr/kaldi), on this project's Bisaya corpus.

**Relationship to `main.ipynb`:** that notebook evaluates a commercial ASR API
(ElevenLabs Scribe) against this same corpus. This notebook instead trains a
model *from scratch*, per the reproduction guide's methodology. They are two
separate, complementary experiments on the same data -- this one does not
depend on `main.ipynb` having been run.

**Environment requirement:** Kaldi does not build on native Windows. This
notebook must run in a Linux environment -- a Kaggle notebook, or WSL2 on your
own machine. Running it in a plain Windows Python kernel will fail at the
Kaldi build step.

**Designed to survive interrupted sessions** (Kaggle's per-session time limit
and weekly GPU quota). Every expensive stage below is written to be safely
rerun after a restart:
- GMM stages (Section 11) are skipped automatically if their `final.mdl`
  already exists -- this is *this notebook's own* resume logic.
- `nnet3` TDNN training (Section 14) resumes from its own last saved
  iteration automatically -- this is Kaldi's built-in behavior, not something
  added here.

The one thing Kaldi does **not** persist for you is the working directory
itself across a killed Kaggle session. Section 2 sets up checkpointing for
that.

## 1. Decisions Made Where the Paper/Guide Leaves Things Unspecified

Following the same convention as the reproduction guide itself: anything below
marked as a *decision* is this notebook's own choice, not a value taken from
the paper. Change these if you have a better-informed choice for your data.

| Area | Decision made here | Why |
|---|---|---|
| Language model toolkit | **KenLM** (`lmplz`/`build_binary`) instead of SRILM | SRILM requires a manual license-gated download from srilm.org and can't be scripted into an automated pipeline; KenLM produces an equivalent ARPA file under a permissive license, and Kaldi's `utils/format_lm.sh` consumes ARPA files regardless of which tool produced them |
| Phoneme set / lexicon | A simple deterministic grapheme-to-phoneme (G2P) mapping (Section 8 below), **not** the paper's PS27/PS35 | PS27/PS35's exact per-word pronunciations aren't given in machine-readable form in the guide (only the phoneme inventory tables). Bisaya orthography is close to phonemic, so a rule-based G2P is a reasonable placeholder, but it is not a faithful PS27/PS35 reproduction -- review the generated lexicon before trusting results built on it |
| Position-dependent phones | Disabled (`--position-dependent-phones false`) | Simplification for a first working pipeline; the paper doesn't state whether it used these |
| HMM topology | Kaldi's standard 3-state-per-phone default | The paper swept 3/4/5-state topologies (guide Section 6) as an experiment; this notebook implements the default only |
| CMVN scope | Per-speaker | Guide Section 14 flags this as unspecified and suggests per-speaker as more standard when enough per-speaker data exists |
| iVector dimensionality | 100-dim | Guide Section 14: not specified by the paper; 100 is the common Kaldi recipe default |
| NN input features | Reuses the same 13-dim MFCC+CMVN computed for the GMM stage, rather than a separate 40-dim "hires" MFCC | Simplification -- most modern nnet3 recipes extract a second, higher-resolution feature set for the neural net; this notebook keeps a single feature set to keep the pipeline shorter to debug on a first pass |
| TDNN architecture | The "symmetric" topology from guide Section 6's table (6 layers x 512 nodes, context [-16,16], no LDA splice layer) | Implemented via the context widths the guide gives for this variant; the LDA-based input splice some recipes use is skipped in favor of directly Appending raw context, to avoid an extra `get_lda.sh` dependency |
| Speed perturbation | Kaldi's standard 3-way (0.9x/1.0x/1.1x) via `utils/data/perturb_data_dir_speed_3way.sh` | The paper's own scheme (random factor uniformly in 1.1-1.25 per file) isn't a standard scriptable Kaldi utility. A custom per-file sox loop implementing the paper's exact range is given as a commented-out alternative in Section 7 |
| N-gram orders built | 2-gram and 3-gram | Matches guide Section 6 ("at least 2-gram and 3-gram"); LM weight is swept at decode time (Section 15) per guide Section 8's reported 1%-25% range |
| Train/test split seed | Fixed, printed at split time (Section 6) | The paper's own seed is lost (guide Section 14) -- this notebook's split is reproducible on rerun, but will not match the paper's specific held-out speakers |

If you already have a properly built PS27/PS35 lexicon, replace Section 8's
`word_to_phones` output with it directly and skip the G2P heuristic. If you
want the paper's 3/4/5-state HMM sweep or its other three TDNN variants
(asymmetric/subsampled/plain DNN), regenerate `data/lang/topo` for the former
and edit the xconfig in Section 13 for the latter -- the guide's Section 6
table gives the exact numbers for both.

## 2. Configuration and Kaggle Checkpoint Persistence

Set the paths below once. `WORK_DIR` holds everything this notebook creates
(`kaldi/`, `data/`, `mfcc/`, `exp/`) so a single tar/restore covers the whole
pipeline state.

**Persistence strategy for Kaggle:**
- Everything lives under `/kaggle/working` while the session is alive.
- `save_checkpoint()` tars the current state to a single archive. Run it after
  each expensive stage completes (a GMM stage, an `nnet3` training chunk,
  etc.) -- not just at the end, since a session can die before you get there.
- Between sessions, **commit the notebook ("Save Version")** so the tarball
  survives as that version's output -- then attach it as an input dataset to
  the next session and call `restore_checkpoint()` before continuing.
- If a mid-session crash (not just a session-end timeout) is a real risk for
  you, also push the tarball to a personal Kaggle Dataset on a shorter
  interval -- see the commented block at the end of the next cell; it needs
  `kaggle.json` credentials set up first.

In [ ]:
import os
import subprocess
import tarfile
from pathlib import Path

IS_KAGGLE = Path("/kaggle").exists()

WORK_DIR = Path("/kaggle/working/asr_train") if IS_KAGGLE else Path("./asr_train").resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)

KALDI_ROOT = WORK_DIR / "kaldi"
DATA_ROOT = WORK_DIR / "data"
MFCC_ROOT = WORK_DIR / "mfcc"
EXP_ROOT = WORK_DIR / "exp"

# Same parquet layout main.ipynb reads. If you only have a "test" split
# locally, point this at wherever your full training split lives (e.g. a
# Kaggle input dataset) before running Section 5.
CORPUS_DIR = Path("data/bisaya_audio")

CHECKPOINT_PATH = Path("/kaggle/working/checkpoint.tar.gz") if IS_KAGGLE else WORK_DIR.parent / "checkpoint.tar.gz"

print(f"IS_KAGGLE   = {IS_KAGGLE}")
print(f"WORK_DIR    = {WORK_DIR}")
print(f"CORPUS_DIR  = {CORPUS_DIR.resolve()}")

In [ ]:
def sh(cmd, cwd=None, check=True, env=None):
    # Streams output live -- used for every Kaldi binary/script invocation
    # below instead of `!` magics, so it behaves the same interactively or
    # when a committed Kaggle version replays the notebook top to bottom.
    print(f"$ {cmd}")
    proc = subprocess.run(cmd, shell=True, cwd=cwd, check=check, executable="/bin/bash", env=env)
    return proc.returncode


def save_checkpoint():
    print(f"Archiving {WORK_DIR} -> {CHECKPOINT_PATH} ...")
    with tarfile.open(CHECKPOINT_PATH, "w:gz") as tar:
        tar.add(WORK_DIR, arcname=WORK_DIR.name)
    size_mb = CHECKPOINT_PATH.stat().st_size / 1e6
    print(f"Done. {size_mb:.1f} MB.")


def restore_checkpoint(archive_path=None):
    archive_path = Path(archive_path) if archive_path else CHECKPOINT_PATH
    print(f"Restoring {archive_path} -> {WORK_DIR.parent} ...")
    with tarfile.open(archive_path, "r:gz") as tar:
        tar.extractall(WORK_DIR.parent)
    print("Done. Re-run the setup/config cells above, then continue -- completed "
          "GMM stages and nnet3 iterations will be detected and skipped automatically.")


# --- Optional: push checkpoints to a personal Kaggle Dataset mid-session ---
# Only needed if mid-session crashes (not just session-end timeouts) are a
# real concern for you; requires the Kaggle API configured (kaggle.json).
#
# KAGGLE_DATASET_SLUG = "your-username/asr-train-checkpoint"
# def push_checkpoint_to_dataset():
#     save_checkpoint()
#     sh(f"kaggle datasets version -p {CHECKPOINT_PATH.parent} -m 'checkpoint update' "
#        f"-d {KAGGLE_DATASET_SLUG}")

## 3. Install Dependencies

Builds Kaldi from source (there is no pip package for it), plus KenLM's
`lmplz`/`build_binary` (Kaldi ships a scriptable installer for this -- no
license gate, unlike SRILM) and `sox` for on-the-fly resampling in `wav.scp`.

**This step is slow** (compiling Kaldi typically takes 30-90 minutes) but it
is naturally resumable: `make` only rebuilds what changed, and the cell below
skips `git clone` if `kaldi/` already exists (e.g. restored from a
checkpoint) and just re-runs `make`, which continues from wherever the
compile left off.

In [ ]:
sh("apt-get update -qq && apt-get install -y -qq "
   "build-essential automake autoconf libtool subversion git zlib1g-dev "
   "gfortran libatlas-base-dev sox")

sh("pip install -q tqdm")

In [ ]:
if not KALDI_ROOT.exists():
    sh(f"git clone --depth 1 https://github.com/kaldi-asr/kaldi.git {KALDI_ROOT}")
else:
    print(f"{KALDI_ROOT} already exists -- skipping clone (resumed from checkpoint).")

# tools/: OpenFST and other bundled dependencies. Incremental -- safe to
# re-run after an interruption, make only redoes unfinished work.
sh("make -j$(nproc)", cwd=KALDI_ROOT / "tools")

# KenLM: Kaldi's bundled installer builds lmplz/build_binary without needing
# SRILM's license-gated download.
sh("extras/install_kenlm.sh", cwd=KALDI_ROOT / "tools")

In [ ]:
src_dir = KALDI_ROOT / "src"

# use-cuda=no: this targets CPU-only environments per the earlier hardware
# discussion (no CUDA-capable GPU locally). If your Kaggle session has a GPU
# attached, drop --use-cuda=no to let nnet3 training use it.
if not (src_dir / "kaldi.mk").exists():
    sh("./configure --shared --use-cuda=no", cwd=src_dir)

sh("make -j$(nproc) depend", cwd=src_dir)
sh("make -j$(nproc)", cwd=src_dir)

print("Kaldi build complete." if (src_dir / "bin" / "compute-mfcc-feats").exists()
      else "WARNING: expected binary not found -- check the build log above for errors.")

## 4. Kaldi Recipe Scaffolding (`path.sh`, `cmd.sh`, `steps/`, `utils/`)

Standard Kaldi recipe layout: a `path.sh` that puts Kaldi's binaries on
`PATH`, a `cmd.sh` that says jobs run locally (`run.pl`) rather than on a
cluster queue (there's no Sun Grid Engine/Slurm here, so this is the only
sane choice on Kaggle or a single workstation), and symlinks to the generic
`steps/`/`utils/` script libraries that ship inside `kaldi/egs/wsj/s5/` and
are reused, unmodified, by every Kaldi recipe.

In [ ]:
path_sh_lines = [
    f"export KALDI_ROOT={KALDI_ROOT}",
    "export PATH=$PWD/utils/:$KALDI_ROOT/tools/openfst/bin:$PWD:$PATH",
    ". $KALDI_ROOT/tools/config/common_path.sh",
    "export LC_ALL=C",
]
(WORK_DIR / "path.sh").write_text("\n".join(path_sh_lines) + "\n")

cmd_sh_lines = [
    "export train_cmd=run.pl",
    "export decode_cmd=run.pl",
    "export mkgraph_cmd=run.pl",
]
(WORK_DIR / "cmd.sh").write_text("\n".join(cmd_sh_lines) + "\n")

wsj_s5 = KALDI_ROOT / "egs" / "wsj" / "s5"
for name in ("steps", "utils"):
    link = WORK_DIR / name
    if not link.exists():
        link.symlink_to(wsj_s5 / name)

print("path.sh, cmd.sh, steps/, utils/ ready under", WORK_DIR)

## 5. Data Preparation: Parquet Corpus -> Kaldi Data Directories

Same parquet layout `main.ipynb` reads (`{split}-000NN-of-000TT.parquet`, with
an `audio` struct column holding raw WAV bytes, plus `transcript` and
`speaker_id`). Loading logic mirrors `main.ipynb`'s Section 3, kept
independent so this notebook doesn't require that one to have been run.

If `CORPUS_DIR` only contains a `test-*.parquet` split (true for this
project's `data/bisaya_audio` as of writing), there isn't enough data here to
train a speaker-independent ASR model -- a handful of speakers' worth of test
data is not a training corpus. Point `CORPUS_DIR` at wherever your full
split(s) live (e.g. a Kaggle input dataset with the corpus's `train` files)
before proceeding.

In [ ]:
import pandas as pd

def load_parquet_split(corpus_dir, glob_pattern):
    files = sorted(Path(corpus_dir).glob(glob_pattern))
    if not files:
        return pd.DataFrame()
    dfs = []
    for file in files:
        temp = pd.read_parquet(file)
        temp["source_file"] = file.name
        dfs.append(temp)
    return pd.concat(dfs, ignore_index=True)


all_files = sorted(CORPUS_DIR.glob("*.parquet"))
splits_present = sorted(set(f.name.split("-")[0] for f in all_files))
print(f"Parquet files found under {CORPUS_DIR}: {len(all_files)}")
print(f"Split prefixes present: {splits_present}")

if splits_present == ["test"]:
    print("\nWARNING: only a 'test' split is present. This is not enough data "
          "to train on -- point CORPUS_DIR at a location with a train split "
          "before continuing (see the markdown note above).")

# Adjust these globs to match whatever split names your corpus actually uses.
train_df = load_parquet_split(CORPUS_DIR, "train*.parquet")
test_df = load_parquet_split(CORPUS_DIR, "test*.parquet")

print(f"\ntrain_df: {len(train_df)} rows, {train_df['speaker_id'].nunique() if len(train_df) else 0} speakers")
print(f"test_df:  {len(test_df)} rows, {test_df['speaker_id'].nunique() if len(test_df) else 0} speakers")

In [ ]:
import re

def kaldi_normalize_text(text):
    # Lowercase + strip punctuation, matching guide Section 3's transcript
    # convention. Deliberately does NOT collapse u/o the way main.ipynb's
    # normalize_bisaya does for WER scoring -- that merge is a scoring-time
    # leniency, not a valid training transcript for a model that's supposed
    # to learn the u/o distinction from audio.
    text = text.lower()
    text = re.sub(r"[^\w\s']", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def build_kaldi_data_dir(df, out_dir, wav_out_dir):
    out_dir = Path(out_dir)
    wav_out_dir = Path(wav_out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    wav_out_dir.mkdir(parents=True, exist_ok=True)

    wav_lines, text_lines, utt2spk_lines = [], [], []

    for i, row in df.iterrows():
        # utt-ids are prefixed with speaker-id -- Kaldi's sort-order
        # conventions require this for utt2spk/spk2utt to line up.
        spk = str(row["speaker_id"])
        utt_id = f"{spk}-{i:06d}"

        wav_path = wav_out_dir / f"{utt_id}.wav"
        wav_path.write_bytes(row["audio"]["bytes"])

        # Piped through sox so every utterance ends up at one consistent
        # sample rate/channel count regardless of the source file's own rate.
        wav_lines.append(f"{utt_id} sox {wav_path} -r 16000 -c 1 -t wav - |")

        transcript = kaldi_normalize_text(str(row["transcript"]))
        text_lines.append(f"{utt_id} {transcript}")
        utt2spk_lines.append(f"{utt_id} {spk}")

    (out_dir / "wav.scp").write_text("\n".join(sorted(wav_lines)) + "\n")
    (out_dir / "text").write_text("\n".join(sorted(text_lines)) + "\n")
    (out_dir / "utt2spk").write_text("\n".join(sorted(utt2spk_lines)) + "\n")

    sh(f"utils/utt2spk_to_spk2utt.pl {out_dir}/utt2spk > {out_dir}/spk2utt", cwd=WORK_DIR)
    sh(f"utils/fix_data_dir.sh {out_dir}", cwd=WORK_DIR)
    sh(f"utils/validate_data_dir.sh --no-feats {out_dir}", cwd=WORK_DIR)

    print(f"{out_dir}: {len(wav_lines)} utterances, {df['speaker_id'].nunique()} speakers")

## 6. Speaker-Independent Train/Test Split

Per guide Section 4: split **by speaker**, not by utterance, ~80/20, with the
split held fixed for every model trained afterward. If `test_df` from Section
5 is already a proper held-out split from your corpus, this cell is skipped
automatically and that split is used directly.

In [ ]:
SPLIT_SEED = 42  # fixed and printed so this split is reproducible on rerun

def speaker_independent_split(df, test_fraction=0.2, seed=SPLIT_SEED):
    speakers = sorted(df["speaker_id"].unique())
    shuffled = pd.Series(speakers).sample(frac=1.0, random_state=seed)
    n_test = max(1, round(len(speakers) * test_fraction))
    test_speakers = set(shuffled.iloc[:n_test])
    train_speakers = set(shuffled.iloc[n_test:])

    assert train_speakers.isdisjoint(test_speakers)

    return df[df["speaker_id"].isin(train_speakers)].copy(), df[df["speaker_id"].isin(test_speakers)].copy()


if len(test_df) == 0 and len(train_df) > 0:
    print("No separate test split found -- deriving a speaker-independent split from train_df.")
    train_df, test_df = speaker_independent_split(train_df)

print(f"seed = {SPLIT_SEED}")
print(f"train: {train_df['speaker_id'].nunique()} speakers, {len(train_df)} utterances")
print(f"test:  {test_df['speaker_id'].nunique()} speakers, {len(test_df)} utterances")
print(f"speaker overlap: {set(train_df['speaker_id']) & set(test_df['speaker_id'])}")

build_kaldi_data_dir(train_df, DATA_ROOT / "train", WORK_DIR / "wav" / "train")
build_kaldi_data_dir(test_df, DATA_ROOT / "test", WORK_DIR / "wav" / "test")

## 7. Speed Perturbation (Training Data Only)

Kaldi's standard 3-way speed perturbation (0.9x/1.0x/1.1x), applied only to
the training set, as a data-augmentation step per guide Section 3. This is
the scriptable, idiomatic-Kaldi version of the paper's augmentation step, not
an exact reproduction of its random-1.1-1.25 scheme (see Section 1's
Decisions table). The paper's exact scheme is given below as a commented-out
alternative if you want to match it precisely instead.

In [ ]:
sh(f"utils/data/perturb_data_dir_speed_3way.sh {DATA_ROOT}/train {DATA_ROOT}/train_sp", cwd=WORK_DIR)
sh(f"utils/fix_data_dir.sh {DATA_ROOT}/train_sp", cwd=WORK_DIR)

TRAIN_DATA_DIR = DATA_ROOT / "train_sp"

# --- Alternative: paper's exact random-1.1-1.25-per-file scheme instead ---
# import random
# random.seed(SPLIT_SEED)
# perturbed_dir = WORK_DIR / "wav" / "train_perturbed"
# perturbed_dir.mkdir(exist_ok=True)
# orig_text = dict(l.split(" ", 1) for l in (DATA_ROOT / "train" / "text").read_text().splitlines())
# for line in (DATA_ROOT / "train" / "wav.scp").read_text().splitlines():
#     utt_id, wav_cmd = line.split(" ", 1)
#     factor = round(random.uniform(1.1, 1.25), 3)
#     new_id = f"{utt_id}-sp{factor}"
#     out_path = perturbed_dir / f"{new_id}.wav"
#     sh(f"{wav_cmd.rstrip(' |')} sox -t wav - {out_path} speed {factor}")
#     # then append new_id lines to copies of wav.scp/text/utt2spk and re-run
#     # utils/fix_data_dir.sh on a merged data/train_custom_sp directory.

## 8. Lexicon and Phoneme Data (`data/local/dict`)

A rule-based grapheme-to-phoneme mapping -- **see Section 1's Decisions
table**: this is a placeholder for the paper's PS27/PS35 pronunciation
dictionaries, which aren't available in machine-readable form. Bisaya/Filipino
orthography is close to phonemic (roughly one letter per sound, plus the `ng`
digraph and a glottal stop marked by an apostrophe), which makes a rule-based
approach reasonable, but review the generated `lexicon.txt` before trusting
downstream results -- words with irregular spelling/pronunciation (loanwords
especially) will get naive, possibly wrong, phone sequences.

In [ ]:
def word_to_phones(word):
    # Each letter is its own phone, except the 'ng' digraph (one phone) and
    # an apostrophe (glottal stop, 'q'). Loanword letters (c, f, j, q, v, x,
    # z) pass through as their own phones too -- rare enough in this domain
    # to review by hand rather than map algorithmically.
    w = re.sub(r"[^a-z']", "", word.lower())
    phones = []
    i = 0
    while i < len(w):
        if w[i:i + 2] == "ng":
            phones.append("ng")
            i += 2
        elif w[i] == "'":
            phones.append("q")
            i += 1
        else:
            phones.append(w[i])
            i += 1
    return phones


def build_lexicon(text_paths, dict_dir):
    dict_dir = Path(dict_dir)
    dict_dir.mkdir(parents=True, exist_ok=True)

    vocab = set()
    for p in text_paths:
        for line in Path(p).read_text().splitlines():
            words = line.split(" ")[1:]
            vocab.update(w for w in words if w)

    lexicon_lines = ["<unk> spn"]
    all_phones = set()
    for word in sorted(vocab):
        phones = word_to_phones(word)
        if not phones:
            continue
        all_phones.update(phones)
        lexicon_lines.append(f"{word} {' '.join(phones)}")

    (dict_dir / "lexicon.txt").write_text("\n".join(lexicon_lines) + "\n")
    (dict_dir / "silence_phones.txt").write_text("sil\nspn\n")
    (dict_dir / "optional_silence.txt").write_text("sil\n")
    (dict_dir / "nonsilence_phones.txt").write_text("\n".join(sorted(all_phones)) + "\n")
    (dict_dir / "extra_questions.txt").write_text("")

    print(f"Vocabulary: {len(vocab)} words, {len(all_phones)} distinct phones -> {dict_dir}")
    return dict_dir


LOCAL_DICT_DIR = build_lexicon(
    [DATA_ROOT / "train" / "text", DATA_ROOT / "test" / "text"],
    DATA_ROOT / "local" / "dict",
)

In [ ]:
sh(f"utils/prepare_lang.sh --position-dependent-phones false "
   f"{LOCAL_DICT_DIR} '<unk>' {DATA_ROOT}/local/lang {DATA_ROOT}/lang", cwd=WORK_DIR)

## 9. Language Model (KenLM -> ARPA -> Kaldi `G.fst`)

Builds 2-gram and 3-gram word-level LMs from the training transcripts
themselves (guide Section 6: in-domain LM, not a general-domain corpus, since
the target domain is narrow). KenLM's `lmplz` produces a standard ARPA file,
which `utils/format_lm.sh` consumes identically to an SRILM-produced one.

In [ ]:
kenlm_bin = KALDI_ROOT / "tools" / "kenlm" / "build" / "bin"
LM_DIR = DATA_ROOT / "local" / "lm"
LM_DIR.mkdir(parents=True, exist_ok=True)

train_text_lines = (DATA_ROOT / "train" / "text").read_text().splitlines()
corpus_txt = LM_DIR / "corpus.txt"
corpus_txt.write_text("\n".join(" ".join(line.split(" ")[1:]) for line in train_text_lines) + "\n")

for order in (2, 3):
    arpa_path = LM_DIR / f"{order}gram.arpa"
    sh(f"{kenlm_bin}/lmplz -o {order} --discount_fallback < {corpus_txt} > {arpa_path}")
    sh(f"utils/format_lm.sh {DATA_ROOT}/lang {arpa_path} {LOCAL_DICT_DIR}/lexicon.txt "
       f"{DATA_ROOT}/lang_{order}g", cwd=WORK_DIR)

print("Built lang_2g and lang_3g graph directories under", DATA_ROOT)

## 10. MFCC + CMVN Feature Extraction

Guide Section 5: 25 ms window, 10 ms frameshift, 13 static coefficients
(deltas are added later, at the model-input stage, not here) -- Kaldi's MFCC
defaults already match this, so no custom `mfcc.conf` is needed. CMVN is
computed **per speaker** (this notebook's documented choice -- see
Section 1).

In [ ]:
N_JOBS = min(8, os.cpu_count() or 4)

for name, data_dir in [("train", TRAIN_DATA_DIR), ("test", DATA_ROOT / "test")]:
    sh(f"steps/make_mfcc.sh --cmd run.pl --nj {N_JOBS} {data_dir} {EXP_ROOT}/make_mfcc/{name} {MFCC_ROOT}", cwd=WORK_DIR)
    sh(f"steps/compute_cmvn_stats.sh {data_dir} {EXP_ROOT}/make_mfcc/{name} {MFCC_ROOT}", cwd=WORK_DIR)
    sh(f"utils/fix_data_dir.sh {data_dir}", cwd=WORK_DIR)

save_checkpoint()

## 11. HMM-GMM Baseline (Monophone -> Triphone -> LDA+MLLT -> SAT)

The standard Kaldi progression from guide Section 6: monophone, then
triphone (delta features), then an LDA+MLLT enhancement, then speaker-adaptive
training (SAT) -- SAT was the paper's best-performing HMM-GMM configuration
for Bisaya (guide Section 10), so it's the one this notebook carries forward
as the alignment source for TDNN training in Section 14.

**Resumability:** `run_stage()` below checks for `final.mdl` in each stage's
output directory before running it, and skips the stage if it's already
there. That's the actual "resume across Kaggle sessions" mechanism for this
whole section -- restore your checkpoint, re-run this cell, and any stage
that finished in a previous session is skipped in seconds; only the
first unfinished stage actually runs.

In [ ]:
def run_stage(name, out_dir, cmd):
    out_dir = Path(out_dir)
    if (out_dir / "final.mdl").exists():
        print(f"[skip] {name}: {out_dir}/final.mdl already exists")
        return
    print(f"[run]  {name}")
    sh(cmd, cwd=WORK_DIR)
    save_checkpoint()


# Monophone
run_stage(
    "mono", EXP_ROOT / "mono",
    f"steps/train_mono.sh --cmd run.pl --nj {N_JOBS} "
    f"{TRAIN_DATA_DIR} {DATA_ROOT}/lang {EXP_ROOT}/mono"
)
sh(f"steps/align_si.sh --cmd run.pl --nj {N_JOBS} "
   f"{TRAIN_DATA_DIR} {DATA_ROOT}/lang {EXP_ROOT}/mono {EXP_ROOT}/mono_ali", cwd=WORK_DIR)

# Triphone (delta + delta-delta features)
run_stage(
    "tri1", EXP_ROOT / "tri1",
    f"steps/train_deltas.sh --cmd run.pl 2000 10000 "
    f"{TRAIN_DATA_DIR} {DATA_ROOT}/lang {EXP_ROOT}/mono_ali {EXP_ROOT}/tri1"
)
sh(f"steps/align_si.sh --cmd run.pl --nj {N_JOBS} "
   f"{TRAIN_DATA_DIR} {DATA_ROOT}/lang {EXP_ROOT}/tri1 {EXP_ROOT}/tri1_ali", cwd=WORK_DIR)

# LDA+MLLT enhancement
run_stage(
    "tri2", EXP_ROOT / "tri2",
    f"steps/train_lda_mllt.sh --cmd run.pl 2500 15000 "
    f"{TRAIN_DATA_DIR} {DATA_ROOT}/lang {EXP_ROOT}/tri1_ali {EXP_ROOT}/tri2"
)
sh(f"steps/align_si.sh --cmd run.pl --nj {N_JOBS} "
   f"{TRAIN_DATA_DIR} {DATA_ROOT}/lang {EXP_ROOT}/tri2 {EXP_ROOT}/tri2_ali", cwd=WORK_DIR)

# SAT -- the paper's best HMM-GMM configuration; final alignments (tri3_ali)
# below are what TDNN training in Section 11 learns to reproduce.
run_stage(
    "tri3", EXP_ROOT / "tri3",
    f"steps/train_sat.sh --cmd run.pl 2500 15000 "
    f"{TRAIN_DATA_DIR} {DATA_ROOT}/lang {EXP_ROOT}/tri2_ali {EXP_ROOT}/tri3"
)
sh(f"steps/align_fmllr.sh --cmd run.pl --nj {N_JOBS} "
   f"{TRAIN_DATA_DIR} {DATA_ROOT}/lang {EXP_ROOT}/tri3 {EXP_ROOT}/tri3_ali", cwd=WORK_DIR)

save_checkpoint()

## 12. GMM Baseline Decode + WER (Sanity Check)

Decodes the test set with the SAT (`tri3`) model before investing time in
TDNN training. This mirrors the guide's own framing (Section 9): HMM-GMM is
the internal baseline the TDNN-HMM is compared against, so getting a number
here first is a useful early checkpoint -- if this comes back badly broken
(e.g. WER near 100%), something upstream (lexicon, data prep) needs fixing
before TDNN training is worth running at all.

In [ ]:
sh(f"utils/mkgraph.sh {DATA_ROOT}/lang_3g {EXP_ROOT}/tri3 {EXP_ROOT}/tri3/graph", cwd=WORK_DIR)

sh(f"steps/decode_fmllr.sh --cmd run.pl --nj {N_JOBS} "
   f"{EXP_ROOT}/tri3/graph {DATA_ROOT}/test {EXP_ROOT}/tri3/decode_test", cwd=WORK_DIR)

sh(f"grep WER {EXP_ROOT}/tri3/decode_test/wer_* | utils/best_wer.sh", cwd=WORK_DIR)

## 13. TDNN-HMM (`nnet3`): iVector Extractor + Network Definition

Trains a diagonal-UBM and iVector extractor (guide Section 5: iVectors are
an additional NN-only input), extracts 100-dim iVectors for both splits, then
defines the TDNN network via an xconfig matching the paper's "symmetric"
topology (guide Section 6): 6 ReLU layers of 512 nodes each, with per-layer
context widths of [-2,2], [-1,1], [-1,1], [-3,3], [-3,3], [-6,6] (overall
context [-16,16]), iVectors appended at the first layer.

In [ ]:
IVECTOR_DIM = 100

run_stage(
    "diag_ubm", EXP_ROOT / "nnet3" / "diag_ubm",
    f"steps/online/nnet2/train_diag_ubm.sh --cmd run.pl --nj {N_JOBS} --num-frames 200000 "
    f"{TRAIN_DATA_DIR} 512 {EXP_ROOT}/tri3 {EXP_ROOT}/nnet3/diag_ubm"
)
run_stage(
    "ivector_extractor", EXP_ROOT / "nnet3" / "extractor",
    f"steps/online/nnet2/train_ivector_extractor.sh --cmd run.pl --nj {N_JOBS} "
    f"--ivector-dim {IVECTOR_DIM} "
    f"{TRAIN_DATA_DIR} {EXP_ROOT}/nnet3/diag_ubm {EXP_ROOT}/nnet3/extractor"
)

for name, data_dir in [("train", TRAIN_DATA_DIR), ("test", DATA_ROOT / "test")]:
    out_dir = EXP_ROOT / "nnet3" / f"ivectors_{name}"
    if not out_dir.exists():
        sh(f"steps/online/nnet2/extract_ivectors_online.sh --cmd run.pl --nj {N_JOBS} "
           f"{data_dir} {EXP_ROOT}/nnet3/extractor {out_dir}", cwd=WORK_DIR)

save_checkpoint()

In [ ]:
TDNN_DIR = EXP_ROOT / "nnet3" / "tdnn"
configs_dir = TDNN_DIR / "configs"
configs_dir.mkdir(parents=True, exist_ok=True)

num_targets = subprocess.run(
    f"tree-info {EXP_ROOT}/tri3_ali/tree | grep num-pdfs | awk '{{print $2}}'",
    shell=True, cwd=WORK_DIR, capture_output=True, text=True, executable="/bin/bash",
    env={**os.environ, "PATH": f"{KALDI_ROOT}/src/bin:" + os.environ["PATH"]},
).stdout.strip()
print(f"num_targets (pdfs) = {num_targets}")

xconfig_lines = [
    "input dim=100 name=ivector",
    "input dim=13 name=input",
    "relu-renorm-layer name=tdnn1 dim=512 input=Append(-2,-1,0,1,2,ReplaceIndex(ivector, t, 0))",
    "relu-renorm-layer name=tdnn2 dim=512 input=Append(-1,0,1)",
    "relu-renorm-layer name=tdnn3 dim=512 input=Append(-1,0,1)",
    "relu-renorm-layer name=tdnn4 dim=512 input=Append(-3,0,3)",
    "relu-renorm-layer name=tdnn5 dim=512 input=Append(-3,0,3)",
    "relu-renorm-layer name=tdnn6 dim=512 input=Append(-6,0,6)",
    f"output-layer name=output dim={num_targets} max-change=1.5",
]
(configs_dir / "network.xconfig").write_text("\n".join(xconfig_lines) + "\n")

sh(f"steps/nnet3/xconfig_to_configs.py "
   f"--xconfig-file {configs_dir}/network.xconfig --config-dir {configs_dir}", cwd=WORK_DIR)

## 14. TDNN-HMM Training

Guide Section 7's only two confirmed hyperparameters: 5 epochs, learning
rate 0.01 -> 0.001. Everything else Kaldi's `nnet3` recipe defaults handle
(guide Section 7 marks optimizer/batch size/weight decay/etc. as
`Not specified in the paper` -- these are `train_dnn.py`'s own defaults here,
not values copied from the paper).

**This is the resumable-by-Kaldi-itself stage** described at the top of this
notebook: if this cell is interrupted, restore your checkpoint and re-run it
unchanged -- `train_dnn.py` scans `TDNN_DIR` for the last completed iteration
and continues from there rather than restarting. `--use-gpu no` matches this
notebook's CPU-only build from Section 3; drop it (and rebuild Kaldi with
CUDA) if you do have a CUDA-capable GPU available.

In [ ]:
sh(
    f"steps/nnet3/train_dnn.py --stage=-10 "
    f"--cmd=run.pl --use-gpu=no "
    f"--feat.cmvn-opts='--norm-means=true --norm-vars=false' "
    f"--trainer.num-epochs 5 "
    f"--trainer.optimization.initial-effective-lrate 0.01 "
    f"--trainer.optimization.final-effective-lrate 0.001 "
    f"--trainer.optimization.num-jobs-initial 1 "
    f"--trainer.optimization.num-jobs-final 1 "
    f"--feat-dir {TRAIN_DATA_DIR} "
    f"--online-ivector-dir {EXP_ROOT}/nnet3/ivectors_train "
    f"--ali-dir {EXP_ROOT}/tri3_ali "
    f"--lang {DATA_ROOT}/lang_3g "
    f"--dir {TDNN_DIR}",
    cwd=WORK_DIR,
)

save_checkpoint()

## 15. Decode + Evaluate the TDNN-HMM Model

Reuses the `tri3` graph from Section 12 (same tree/lexicon the TDNN was
aligned against) for decoding, sweeping the LM weight over roughly the
paper's reported 1%-25% range (guide Section 8) via `--min-lmwt`/
`--max-lmwt`, then prints the best WER found -- directly comparable to the
GMM baseline from Section 12 and to guide Section 10's reference numbers
(with the caveats from that section: different dataset, so don't expect the
same absolute values).

In [ ]:
sh(
    f"steps/nnet3/decode.sh --cmd run.pl --nj {N_JOBS} "
    f"--online-ivector-dir {EXP_ROOT}/nnet3/ivectors_test "
    f"--min-lmwt 1 --max-lmwt 25 "
    f"{EXP_ROOT}/tri3/graph {DATA_ROOT}/test {TDNN_DIR}/decode_test",
    cwd=WORK_DIR,
)

sh(f"grep WER {TDNN_DIR}/decode_test/wer_* | utils/best_wer.sh", cwd=WORK_DIR)

save_checkpoint()

## 16. Summary

Compare the GMM baseline (Section 12) and TDNN-HMM (Section 15) best-WER
lines printed above against guide Section 10's reference table. Per that
section's own caveat: this corpus differs from the paper's in vocabulary
size, domain, speaker count, and total duration, so treat any gap to the
paper's numbers as expected rather than a sign of a broken pipeline -- and
treat Section 8's placeholder G2P lexicon as the most likely source of a
*larger-than-expected* gap specifically, since it's the piece with the least
grounding in the paper's actual methodology.